# Build a Transformer from Scratch — the Karpathy way (Part 1: fundamentals)

This is a **step-by-step, from-zero** build of a Transformer language model, following the structure
of Andrej Karpathy's *"Let's build GPT: from scratch, in code, spelled out."*

We start with the simplest possible model and add **one idea at a time**, so that by the end you
understand *why* each part of a Transformer exists. We train a tiny **character-level** model on
text, exactly like the video.

> This is **Part 1: fundamentals**. Once these mechanics click, a follow-up notebook will map them
> onto the **SmolVLA** architecture (`SmolVLA.pdf` in this repo). For now, forget robots — just learn
> the Transformer.

### The plan (each step builds on the last)
1. Get a text dataset and look at it
2. **Tokenize** at the character level (encode/decode)
3. Batching: `block_size`, `batch_size`, and next-token targets
4. **Bigram** baseline — the training/eval/generate skeleton with *no* attention
5. The **mathematical trick** at the heart of self-attention (averaging the past 3 ways)
6. A **single self-attention head**
7. **Multi-head** attention
8. **Feed-forward** network
9. **Blocks**: residual connections + LayerNorm
10. Assemble the full **GPT**, train it, and generate text
11. A short bridge to what comes next (SmolVLA)

Run the cells top to bottom. A GPU (Colab: *Runtime → Change runtime type → GPU*) makes step 10 fast,
but everything runs on CPU too (just slower).


In [37]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)                     # same seed Karpathy uses, for reproducibility
# Use GPU if available — training is much faster; otherwise fall back to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

torch 2.6.0+cu124 | device: cuda


## 1. Get the data

We use the classic **tiny Shakespeare** corpus (~1 MB of text). We try to download it; if there's no
network, we fall back to a small embedded sample so the notebook still runs.


In [38]:
import urllib.request

# URL for the full tiny Shakespeare dataset (~1 MB of raw text)
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
# A small hardcoded excerpt used when the download fails (e.g. no internet)
FALLBACK = (
    "First Citizen:\nBefore we proceed any further, hear me speak.\n\n"
    "All:\nSpeak, speak.\n\n"
    "First Citizen:\nYou are all resolved rather to die than to famish?\n\n"
    "All:\nResolved. resolved.\n\n"
    "First Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\n"
)
try:
    text = urllib.request.urlopen(URL, timeout=15).read().decode("utf-8")
    print("downloaded tiny shakespeare")
except Exception as e:
    print("download failed (", e, ") -> using small embedded fallback")
    text = FALLBACK * 400        # repeat the fallback many times so there's enough data to train on

print("length of dataset in characters:", len(text))
print("----- first 250 chars -----")
print(text[:250])

downloaded tiny shakespeare
length of dataset in characters: 1115394
----- first 250 chars -----
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



## 2. Tokenize at the character level

The model can't read text — it reads **integers**. The simplest tokenizer maps each unique
*character* to an integer id. We build `stoi` (string→int) and `itos` (int→string) lookup tables,
then `encode`/`decode`.

(Real models use *subword* tokenizers like BPE with ~50k tokens; char-level keeps the vocab tiny and
the idea crystal clear.)


In [39]:
# Collect all unique characters and sort them to form the vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
print("vocab size:", vocab_size)
print("vocabulary:", "".join(chars))

# stoi: maps each character to a unique integer index
stoi = {ch: i for i, ch in enumerate(chars)}
# itos: reverse mapping from integer index back to character
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]           # string -> list[int]
decode = lambda l: "".join(itos[i] for i in l)    # list[int] -> string

# Quick round-trip test: encode then decode should give the original string back
print(encode("hello"))
print(decode(encode("hello")))

vocab size: 65
vocabulary: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
[46, 43, 50, 50, 53]
hello


In [40]:
# Encode the whole dataset into one long tensor of integers
data = torch.tensor(encode(text), dtype=torch.long)
# Split 90/10 into training and validation sets
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print("train:", train_data.shape, "| val:", val_data.shape)
print("first 40 tokens:", train_data[:40].tolist())

train: torch.Size([1003854]) | val: torch.Size([111540])
first 40 tokens: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56, 43, 1, 61, 43, 1, 54, 56, 53, 41, 43, 43, 42, 1, 39, 52, 63, 1, 44, 59, 56]


## 3. Batching: context length and next-token targets

A language model is trained to **predict the next token given all previous tokens**. We chop the data
into chunks of length `block_size` (the maximum context). Within a chunk of length `T`, there are `T`
training examples packed together: predict token 1 from token 0, token 2 from tokens 0–1, and so on.

`get_batch` grabs `batch_size` random chunks and returns:
- `x`: the context, shape `(batch_size, block_size)`
- `y`: the same sequence shifted left by one (the targets), same shape


In [41]:
block_size = 32       # max context length (Karpathy's video uses 8, then 256; 32 is a good middle)
batch_size = 16       # independent sequences processed in parallel

def get_batch(split):
    """Sample a random batch of (input, target) chunks from the dataset."""
    d = train_data if split == "train" else val_data
    # Pick batch_size random starting indices (each chunk must fit block_size tokens)
    ix = torch.randint(len(d) - block_size, (batch_size,))
    # x: input context windows, each of length block_size
    x = torch.stack([d[i : i + block_size] for i in ix])
    # y: targets — the same windows shifted right by 1 (next token for each position)
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("x shape:", xb.shape, "| y shape:", yb.shape)
# Show how each prefix of the chunk maps to a next-token prediction target
print("\nWhat 'predict the next token' means, unpacked for one chunk:")
for t in range(6):
    context = xb[0, : t + 1].tolist()
    target = yb[0, t].item()
    print(f"  context {context}  ->  target {target}")

x shape: torch.Size([16, 32]) | y shape: torch.Size([16, 32])

What 'predict the next token' means, unpacked for one chunk:
  context [58]  ->  target 53
  context [58, 53]  ->  target 1
  context [58, 53, 1]  ->  target 41
  context [58, 53, 1, 41]  ->  target 53
  context [58, 53, 1, 41, 53]  ->  target 56
  context [58, 53, 1, 41, 53, 56]  ->  target 56


## 4. Bigram baseline — the whole skeleton, no attention yet

Before any attention, build the simplest model: a **bigram** language model. Each token id directly
looks up a row of logits over the next token via an `nn.Embedding`. It literally only uses the *last*
token to predict the next — no context, no attention. It's a bad model, but it lets us stand up the
**exact training / evaluation / generation loop** we'll reuse for the real Transformer.

Note `generate()` only ever uses the last time step's logits — that's the autoregressive loop.


In [42]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Each token id directly indexes a row of logits for the next token.
        # This is the entire "model" — no hidden layers, no attention.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx shape: (B, T) — batch of token-index sequences
        logits = self.token_embedding_table(idx)          # (B, T, vocab_size)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            # Flatten (B, T, C) -> (B*T, C) so cross_entropy gets a 2-D input
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        """Autoregressively generate max_new_tokens new tokens."""
        for _ in range(max_new_tokens):
            logits, _ = self(idx)                          # (B, T, vocab)
            logits = logits[:, -1, :]                      # last time-step only -> (B, vocab)
            probs = F.softmax(logits, dim=-1)              # convert to probabilities
            idx_next = torch.multinomial(probs, 1)         # sample one token per batch element
            idx = torch.cat([idx, idx_next], dim=1)        # append to the running sequence
        return idx

m = BigramLanguageModel(vocab_size).to(device)
logits, loss = m(xb, yb)
print("logits:", logits.shape, "| initial loss:", loss.item())
# Sanity check: a random model guessing uniformly should have loss ≈ ln(vocab_size)
print("(a random model should have loss ~= ln(vocab_size) =", round(torch.log(torch.tensor(vocab_size*1.0)).item(), 3), ")")

logits: torch.Size([16, 32, 65]) | initial loss: 4.605658531188965
(a random model should have loss ~= ln(vocab_size) = 4.174 )


In [43]:
# A small reusable helper to estimate loss on train/val without gradient noise.
# Averages over many batches for a stable estimate.
@torch.no_grad()
def estimate_loss(model, eval_iters=100):
    model.eval()                            # switch to eval mode (disables dropout)
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()   # average over all eval batches
    model.train()                           # switch back to training mode
    return out

# Train the bigram model briefly.
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-2)   # AdamW: Adam with weight decay fix
for step in range(2000):
    xb, yb = get_batch("train")            # sample a fresh random batch each step
    _, loss = m(xb, yb)                    # forward pass -> compute loss
    optimizer.zero_grad(set_to_none=True)  # reset gradients (set_to_none is slightly faster)
    loss.backward()                        # backward pass -> compute gradients
    optimizer.step()                       # update parameters
print("bigram loss after training:", estimate_loss(m))

# Generate from the trained bigram model (starting from a single newline token = index 0).
start = torch.zeros((1, 1), dtype=torch.long, device=device)
print("----- bigram sample -----")
print(decode(m.generate(start, max_new_tokens=200)[0].tolist()))

bigram loss after training: {'train': 2.4603097438812256, 'val': 2.4935882091522217}
----- bigram sample -----



CExthy brid owindakis by blKEY: set bobe d e.
S:
O:3 my d?
LUCous:
Wanthar u qur, t.
War dilasoate awice my.

Hastacom oroup
Yowhthetof is h ble mil ndill, ath iree senghin lat Heridrovets, and Win 


The bigram sample is gibberish with the right *character statistics* — it never looks back more than
one token. To do better, a token must be able to **gather information from previous tokens**. That is
exactly what self-attention provides. Next we derive it.


## 5. The mathematical trick at the heart of self-attention

Goal: let each position `t` build a representation from **all positions `≤ t`** (it must not see the
future). Start with the crudest version of "gather from the past": just **average** the previous
tokens. We'll compute this same average three ways — the third way generalizes into attention.


In [44]:
# Toy tensor: batch=1, time=8, channels=2
torch.manual_seed(1337)
B, T, C = 1, 8, 2
x = torch.randn(B, T, C)

# --- Version 1: explicit loop. xbow[t] = mean of x[0..t]  ('bow' = bag of words) ---
xbow = torch.zeros(B, T, C)
for b in range(B):
    for t in range(T):
        xbow[b, t] = x[b, : t + 1].mean(dim=0)

# --- Version 2: the same average as a matrix multiply with a normalized lower-triangular matrix ---
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(dim=1, keepdim=True)      # each row averages the tokens up to and including t
xbow2 = wei @ x                               # (T,T) @ (B,T,C) -> (B,T,C)

print("weight matrix (row t averages cols 0..t):")
print(wei)
print("versions 1 and 2 match:", torch.allclose(xbow, xbow2))

weight matrix (row t averages cols 0..t):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
versions 1 and 2 match: True


In [45]:
# --- Version 3: produce those weights with softmax over masked scores ---
# This is the form self-attention uses: start from scores, forbid the future with -inf, softmax.
tril = torch.tril(torch.ones(T, T))
scores = torch.zeros(T, T)                     # here scores are all 0 -> uniform average
scores = scores.masked_fill(tril == 0, float("-inf"))   # block the future
wei3 = F.softmax(scores, dim=-1)
xbow3 = wei3 @ x
print("version 3 matches:", torch.allclose(xbow, xbow3))
print("\nThe key idea: those weights don't have to be uniform.\n"
      "If the scores are *computed from the data*, each token can decide how much to pull\n"
      "from each earlier token. That data-dependent weighting IS self-attention.")

version 3 matches: True

The key idea: those weights don't have to be uniform.
If the scores are *computed from the data*, each token can decide how much to pull
from each earlier token. That data-dependent weighting IS self-attention.


## 6. A single self-attention head

Now make the weights data-dependent. Every token emits:
- a **query** ("what am I looking for?")
- a **key**   ("what do I contain?")
- a **value** ("what will I share if attended to?")

The affinity between position `t` and position `s` is `query_t · key_s`. We mask the future, softmax,
and use the weights to average the **values**. We also scale by `1/sqrt(head_size)` so the softmax
doesn't saturate (Karpathy's "scaled" attention).


In [46]:
# Standalone demo: self-attention on a toy example (not inside a class yet)
torch.manual_seed(1337)
B, T, C = 4, 8, 32                          # batch=4, time=8, channels=32
x = torch.randn(B, T, C)

head_size = 16
key   = nn.Linear(C, head_size, bias=False)  # projects input into "what do I contain?" space
query = nn.Linear(C, head_size, bias=False)  # projects input into "what am I looking for?" space
value = nn.Linear(C, head_size, bias=False)  # projects input into "what will I share?" space

k = key(x)        # (B, T, head_size) — key vectors for every token
q = query(x)      # (B, T, head_size) — query vectors for every token
# Compute scaled dot-product affinities: how much each query matches each key.
# Dividing by sqrt(head_size) prevents dot products from growing too large,
# which would push softmax into saturated regions with near-zero gradients.
wei = q @ k.transpose(-2, -1) * head_size ** -0.5   # (B, T, T) scaled affinities

# Build a lower-triangular mask and set future positions to -inf.
# After softmax, -inf becomes 0 — so token t cannot attend to tokens > t.
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))     # causal mask: no peeking ahead
wei = F.softmax(wei, dim=-1)                        # normalize each row to a probability distribution

v = value(x)      # (B, T, head_size) — value vectors (the "payload" each token offers)
out = wei @ v     # (B, T, head_size) — weighted sum of values = the attention output

print("output shape:", tuple(out.shape))
print("\nattention weights for one example (row t attends over cols 0..t):")
print(wei[0].round(decimals=2))

output shape: (4, 8, 16)

attention weights for one example (row t attends over cols 0..t):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4000, 0.6000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3100, 0.2900, 0.4000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3200, 0.2200, 0.2400, 0.2100, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1500, 0.2000, 0.1700, 0.1500, 0.3400, 0.0000, 0.0000, 0.0000],
        [0.1300, 0.2500, 0.1300, 0.1100, 0.3100, 0.0700, 0.0000, 0.0000],
        [0.1600, 0.2000, 0.1100, 0.1100, 0.1400, 0.1700, 0.1100, 0.0000],
        [0.0800, 0.1200, 0.1100, 0.1500, 0.1100, 0.1100, 0.1600, 0.1600]],
       grad_fn=<RoundBackward1>)


Notice the weights are now **different per row and per example** — data-dependent — and strictly
lower-triangular (causal). That single block is the entire innovation. Everything else in a
Transformer is plumbing that makes stacks of these trainable. Let's package it.


In [47]:
class Head(nn.Module):
    """One head of causal self-attention."""
    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        # Three learned projections: query, key, value (no bias, following the original paper)
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # Pre-compute a lower-triangular causal mask (not a learned parameter, just a buffer).
        # Registered as a buffer so it moves to GPU with the model and is saved in state_dict.
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        # Dropout applied to the attention weights to regularize which tokens attend to which
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape                                        # batch, time, channels
        k = self.key(x)                                           # (B, T, head_size) — "what do I contain?"
        q = self.query(x)                                         # (B, T, head_size) — "what am I looking for?"
        # Scaled dot-product attention scores: each query dot-products with every key.
        # Scale by 1/sqrt(head_size) to keep variance ~1 and prevent softmax saturation.
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5      # (B, T, T)
        # Causal mask: set future positions to -inf so softmax gives them weight 0.
        # Slice [:T, :T] because the actual sequence may be shorter than block_size.
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)                              # normalize to a probability distribution per row
        wei = self.dropout(wei)                                   # randomly drop some attention edges during training
        v = self.value(x)                                         # (B, T, head_size) — "what do I share?"
        return wei @ v                                            # (B, T, head_size) — weighted sum of values

## 7. Multi-head attention

One head captures one kind of relationship. **Multi-head** attention runs several heads in parallel
(each in a smaller subspace), concatenates their outputs, and projects back. This lets the model
attend to different things at once (e.g. one head tracks the previous vowel, another tracks quotes).


In [48]:
class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention running in parallel."""
    def __init__(self, n_heads, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        # Create n_heads independent attention heads, each operating in a head_size subspace
        self.heads = nn.ModuleList(
            [Head(n_embd, head_size, block_size, dropout) for _ in range(n_heads)]
        )
        # Linear projection to mix the concatenated head outputs back to n_embd dimensions
        self.proj = nn.Linear(n_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Run all heads in parallel and concatenate along the channel dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # (B, T, n_heads*head_size)
        # Project concatenated output back to n_embd and apply dropout
        return self.dropout(self.proj(out))

## 8. Feed-forward network

Attention lets tokens **communicate**. But after gathering information, each token needs to
**process** it individually. That's the per-position feed-forward MLP: expand → nonlinearity →
contract. It runs on each position independently.


In [49]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),   # expand (the 4x is the Transformer convention)
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),   # project back
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

## 9. Blocks: residual connections + LayerNorm

A Transformer **block** = multi-head attention (communicate) + feed-forward (compute). Stacking many
raw blocks won't train well, so we add the two tricks that make depth work:

- **Residual connections** (`x = x + sublayer(x)`): give gradients a straight highway to flow back.
- **LayerNorm**: normalize each token's features. We use the modern **pre-norm** placement — normalize
  *before* each sublayer (`x + sublayer(ln(x))`), which is more stable than the original design.


In [50]:
class Block(nn.Module):
    """One Transformer block: communication (attention) then computation (FFN)."""
    def __init__(self, n_embd, n_heads, block_size, dropout=0.0):
        super().__init__()
        # Each head operates in a subspace of size n_embd // n_heads
        head_size = n_embd // n_heads
        self.sa = MultiHeadAttention(n_heads, n_embd, head_size, block_size, dropout)
        self.ff = FeedForward(n_embd, dropout)
        # LayerNorm before each sub-layer (pre-norm formulation — more stable than post-norm)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Residual connection around attention: x flows straight through + attended info
        x = x + self.sa(self.ln1(x))     # communicate (with residual)
        # Residual connection around FFN: x flows straight through + processed info
        x = x + self.ff(self.ln2(x))     # compute     (with residual)
        return x

## 10. Assemble the full GPT — and train it

Now stack it all:
- a **token embedding** table (id → vector) and a **position embedding** table (Karpathy uses a
  learned one rather than sinusoids),
- a stack of `n_layer` Blocks,
- a final LayerNorm and a linear head back to vocab-size logits.

Then reuse the exact same training loop and `generate()` from the bigram model.


In [51]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, n_heads, n_layer, block_size, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        # Token embedding: maps each integer token id to a dense vector of size n_embd
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # Position embedding: learned vector for each position 0..block_size-1,
        # so the model knows *where* each token sits in the sequence
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # Stack of Transformer blocks — the core of the model.
        # Each block has multi-head attention (communicate) + FFN (compute).
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_heads, block_size, dropout) for _ in range(n_layer)]
        )
        # Final LayerNorm before the output projection (stabilises the last block's output)
        self.ln_f = nn.LayerNorm(n_embd)
        # Linear head: project from n_embd back to vocab_size to produce next-token logits
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding_table(idx)                       # (B, T, n_embd)
        # Create position indices [0, 1, ..., T-1] on the same device as the input
        pos = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, n_embd)
        # Add token and position embeddings element-wise — this is how the model
        # combines "what token is this?" with "where does it appear?"
        x = tok + pos
        x = self.blocks(x)                                         # pass through all Transformer blocks
        x = self.ln_f(x)                                           # final layer norm
        logits = self.lm_head(x)                                   # (B, T, vocab_size)
        loss = None
        if targets is not None:
            B, T, Cv = logits.shape
            # Reshape for cross_entropy: (B*T, vocab_size) logits vs (B*T,) targets
            loss = F.cross_entropy(logits.view(B * T, Cv), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        """Autoregressively append max_new_tokens tokens to the sequence."""
        for _ in range(max_new_tokens):
            # Crop context to the last block_size tokens (the model can't handle longer)
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)                              # forward pass
            logits = logits[:, -1, :]                               # take only the last time step -> (B, vocab)
            probs = F.softmax(logits, dim=-1)                       # convert to probability distribution
            idx_next = torch.multinomial(probs, 1)                  # sample next token
            idx = torch.cat([idx, idx_next], dim=1)                 # append to running sequence
        return idx

In [52]:
# Hyperparameters — small so it trains quickly. Bump these up on a GPU for better samples.
n_embd, n_heads, n_layer, dropout = 64, 4, 4, 0.1
max_iters, eval_interval, lr = 3000, 500, 3e-4

model = GPTLanguageModel(vocab_size, n_embd, n_heads, n_layer, block_size, dropout).to(device)
# Count total trainable parameters (divide by 1000 for readability)
print("parameters:", sum(p.numel() for p in model.parameters()) / 1e3, "K")

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)  # AdamW: Adam with decoupled weight decay
for it in range(max_iters + 1):
    # Periodically evaluate on train & val to track progress and detect overfitting
    if it % eval_interval == 0:
        losses = estimate_loss(model, eval_iters=50)
        print(f"step {it:4d} | train {losses['train']:.3f} | val {losses['val']:.3f}")
    xb, yb = get_batch("train")              # sample a fresh random batch each step
    _, loss = model(xb, yb)                  # forward pass -> compute cross-entropy loss
    optimizer.zero_grad(set_to_none=True)    # reset gradients (set_to_none is slightly faster than zero_grad())
    loss.backward()                          # backward pass -> compute gradients for all parameters
    optimizer.step()                         # update parameters using AdamW

parameters: 209.729 K
step    0 | train 4.325 | val 4.325
step  500 | train 2.518 | val 2.523
step 1000 | train 2.347 | val 2.358
step 1500 | train 2.269 | val 2.286
step 2000 | train 2.188 | val 2.188
step 2500 | train 2.105 | val 2.157
step 3000 | train 2.079 | val 2.113


In [53]:
# Generate! (Small model + tiny training, so expect Shakespeare-flavored gibberish that has learned
# words, spacing, capitalization, and 'Name:' dialogue structure — a huge step up from the bigram.)
start = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(start, max_new_tokens=500)[0].tolist()))


To noust:
And if hat ing in he wit ger-d a me,
And ghtaid naw sI rome, mumm.
Loblive oun, te an the olsto whall yourgest to pelum men ain,
Vrows I my cond, lim, what ty wrothe tou and on berreden he der
Gear, I dis if dovis
Ing to thave dow? I pahis thy so destlarre!
Thow may atust, brut feds, man withtle!

And mow sthire ond tho to the toll,
Tbleess man, for yow paist 'is you hy bintins I ga'd to of thee,
HERICHA for:
Mold hefrThere hale thas thip dreserie shero shey ing firtk not merch ive and


## 11. You just built a GPT. What you now understand

- **Tokenization**: text ↔ integers (char-level here; subword/BPE in real models).
- **Self-attention**: data-dependent, causally-masked weighted averaging via Q·K → softmax → ·V.
- **Multi-head**: several attention subspaces in parallel.
- **Blocks**: attention (communicate) + FFN (compute), glued together by **residuals** and
  **LayerNorm** so deep stacks train.
- **The full loop**: embed → blocks → project to logits → cross-entropy → AdamW → autoregressive
  `generate()`.

### Where to go next
- **Scale it up** (Karpathy's finale): `block_size=256, n_embd=384, n_heads=6, n_layer=6`, more iters,
  on a GPU → recognizable Shakespeare.
- **Part 2 (this repo): connect to SmolVLA** — now built in `02_smolvla_action_expert.ipynb`.
   The same attention you just built is the engine of
  SmolVLA's **action expert** (`SmolVLA.pdf`, §3.1). The differences to look for next:
  - it predicts **continuous actions** with a **flow-matching** (regression) objective, not softmax
    over a vocabulary;
  - it adds **cross-attention** so action tokens read **VLM (vision-language) features**, interleaved
    with the **causal self-attention** you built here;
  - inputs are images + a language instruction + a robot **state token**, not characters.

### Exercises
1. Print the attention weights of `Head` for a real batch and read off what a head has learned.
2. Swap the learned position embedding for the sinusoidal one and compare.
3. Turn dropout off — does the val loss get worse (overfitting)?
4. Change the dataset (paste your own text into `FALLBACK`) and retrain.


## Reference — the original Transformer, and the slice you just built

Here is the full architecture from *Attention Is All You Need* (Vaswani et&#160;al., 2017, Fig.&#160;1),
recreated so it renders anywhere. Read each stack **bottom-to-top**.

<div style="overflow-x:auto">

<svg viewBox="0 0 1080 1230" width="100%" style="max-width:760px;height:auto;display:block;margin:auto" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="The original Transformer architecture (Vaswani et al. 2017, Figure 1)">
<rect x="0" y="0" width="1080" height="1230" rx="16" fill="#FBFAF6"/>
<defs><marker id="ao" viewBox="0 0 8 8" refX="6.5" refY="4" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L7,4 L0,8 z" fill="#4A463E"/></marker></defs>
<path d="M300,1103 V700" fill="none" stroke="#4A463E" stroke-width="1.8"/>
<path d="M780,1103 V443" fill="none" stroke="#4A463E" stroke-width="1.8" marker-end="url(#ao)"/>
<circle cx="300" cy="1050" r="8" fill="#FBFAF6" stroke="#4A463E" stroke-width="1.4"/><text x="300" y="1054" text-anchor="middle" font-size="13" fill="#33302A" font-family="Georgia,Times New Roman,serif">+</text>
<text x="348" y="1054" text-anchor="start" font-size="10.5" fill="#6b6456" font-family="Georgia,Times New Roman,serif">Positional Encoding</text>
<circle cx="780" cy="1050" r="8" fill="#FBFAF6" stroke="#4A463E" stroke-width="1.4"/><text x="780" y="1054" text-anchor="middle" font-size="13" fill="#33302A" font-family="Georgia,Times New Roman,serif">+</text>
<text x="828" y="1054" text-anchor="start" font-size="10.5" fill="#6b6456" font-family="Georgia,Times New Roman,serif">Positional Encoding</text>
<rect x="120" y="758" width="360" height="288" rx="12" fill="none" stroke="#B4AEA1" stroke-width="1.3" stroke-dasharray="5 5"/>
<rect x="600" y="618" width="360" height="428" rx="12" fill="none" stroke="#B4AEA1" stroke-width="1.3" stroke-dasharray="5 5"/>
<text x="300" y="750" text-anchor="middle" font-size="13" fill="#7C7566" font-style="italic" font-family="Georgia,Times New Roman,serif">Encoder</text>
<text x="780" y="610" text-anchor="middle" font-size="13" fill="#7C7566" font-style="italic" font-family="Georgia,Times New Roman,serif">Decoder</text>
<path d="M112,766 h-8 V1038 h8" fill="none" stroke="#B4AEA1" stroke-width="1.3"/>
<text x="72" y="902" transform="rotate(-90 72 902)" text-anchor="middle" font-size="14" fill="#6b6456" font-style="italic" font-family="Georgia,Times New Roman,serif">N &#215;</text>
<path d="M968,626 h8 V1038 h-8" fill="none" stroke="#B4AEA1" stroke-width="1.3"/>
<text x="1004" y="832" transform="rotate(90 1004 832)" text-anchor="middle" font-size="14" fill="#6b6456" font-style="italic" font-family="Georgia,Times New Roman,serif">&#215; N</text>
<rect x="150" y="1103" width="300" height="46" rx="8" fill="#F6CFC9" stroke="#CE8E86" stroke-width="1.4"/><text x="300" y="1130" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Input Embedding</text>
<text x="300" y="1178" text-anchor="middle" font-size="13" fill="#33302A" font-style="italic" font-family="Georgia,Times New Roman,serif">Inputs</text>
<rect x="150" y="979" width="300" height="42" rx="8" fill="#F8C98C" stroke="#CF9445" stroke-width="1.4"/><text x="300" y="1000" text-anchor="middle" font-size="13.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Multi-Head Attention</text><text x="300" y="1015" text-anchor="middle" font-size="10" fill="#6b6456" font-family="Georgia,Times New Roman,serif">Q, K, V all from encoder</text>
<rect x="150" y="909" width="300" height="42" rx="8" fill="#FBF0A6" stroke="#CBB85A" stroke-width="1.4"/><text x="300" y="935" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Add &amp; Norm</text>
<rect x="150" y="839" width="300" height="42" rx="8" fill="#C6DCEF" stroke="#7AA6C9" stroke-width="1.4"/><text x="300" y="865" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Feed Forward</text>
<rect x="150" y="769" width="300" height="42" rx="8" fill="#FBF0A6" stroke="#CBB85A" stroke-width="1.4"/><text x="300" y="795" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Add &amp; Norm</text>
<rect x="630" y="1103" width="300" height="46" rx="8" fill="#F6CFC9" stroke="#CE8E86" stroke-width="1.4"/><text x="780" y="1130" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Output Embedding</text>
<text x="780" y="1178" text-anchor="middle" font-size="13" fill="#33302A" font-style="italic" font-family="Georgia,Times New Roman,serif">Outputs (shifted right)</text>
<rect x="630" y="979" width="300" height="42" rx="8" fill="#F8C98C" stroke="#CF9445" stroke-width="1.4"/><text x="780" y="1000" text-anchor="middle" font-size="13" fill="#33302A" font-family="Georgia,Times New Roman,serif">Masked Multi-Head Attention</text><text x="780" y="1015" text-anchor="middle" font-size="10" fill="#6b6456" font-family="Georgia,Times New Roman,serif">Q, K, V from decoder (causal)</text>
<rect x="630" y="909" width="300" height="42" rx="8" fill="#FBF0A6" stroke="#CBB85A" stroke-width="1.4"/><text x="780" y="935" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Add &amp; Norm</text>
<rect x="630" y="839" width="300" height="42" rx="8" fill="#F8C98C" stroke="#CF9445" stroke-width="1.4"/><text x="780" y="858" text-anchor="middle" font-size="13.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Multi-Head Attention</text><text x="780" y="873" text-anchor="middle" font-size="10" fill="#B5502E" font-family="Georgia,Times New Roman,serif">Q: decoder    K, V: encoder</text>
<rect x="630" y="769" width="300" height="42" rx="8" fill="#FBF0A6" stroke="#CBB85A" stroke-width="1.4"/><text x="780" y="795" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Add &amp; Norm</text>
<rect x="630" y="699" width="300" height="42" rx="8" fill="#C6DCEF" stroke="#7AA6C9" stroke-width="1.4"/><text x="780" y="725" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Feed Forward</text>
<rect x="630" y="629" width="300" height="42" rx="8" fill="#FBF0A6" stroke="#CBB85A" stroke-width="1.4"/><text x="780" y="655" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Add &amp; Norm</text>
<rect x="655" y="539" width="250" height="42" rx="8" fill="#D9CBE9" stroke="#A08CC0" stroke-width="1.4"/><text x="780" y="565" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Linear</text>
<rect x="655" y="464" width="250" height="42" rx="8" fill="#C8E6C0" stroke="#77AE6C" stroke-width="1.4"/><text x="780" y="490" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Softmax</text>
<rect x="645" y="397" width="270" height="46" rx="8" fill="#E7E3DA" stroke="#B4AEA1" stroke-width="1.4"/><text x="780" y="424" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Output Probabilities</text>
<path d="M300,700 H556 V830 H628" fill="none" stroke="#B5502E" stroke-width="1.8" marker-end="url(#ao)"/>
<path d="M300,715 H572 V848 H628" fill="none" stroke="#B5502E" stroke-width="1.8" marker-end="url(#ao)"/>
<text x="486" y="692" text-anchor="middle" font-size="11.5" fill="#B5502E" font-style="italic" font-family="Georgia,Times New Roman,serif">encoder output</text>
<text x="486" y="760" text-anchor="middle" font-size="11.5" fill="#B5502E" font-style="italic" font-family="Georgia,Times New Roman,serif">&#8594; K, V</text>
</svg>

</div>

*Legend: pink = embedding, orange = attention, yellow = Add&nbsp;&amp;&nbsp;Norm, blue = feed-forward,
purple = Linear, green = Softmax.*

**Where your GPT sits in this picture.** You built the **right (decoder) stack, with the middle
attention box deleted** &mdash; i.e. `Masked Multi-Head Attention` &rarr; `Add & Norm` &rarr;
`Feed Forward` &rarr; `Add & Norm`, repeated `N`&#215;, then `Linear` &rarr; `Softmax`. There is **no
encoder** and **no cross-attention**, because a language model just continues its own text &mdash; there
is no separate input sequence to attend to. That "decoder-only" simplification is what GPT, LLaMA, and
friends all use.

**The three attention boxes differ only in where Q, K, V come from** (the operation inside is identical
&mdash; the `Head` you wrote in &sect;6):

| box | Q from | K, V from | masked? | role |
|---|---|---|---|---|
| encoder self-attn | encoder | encoder | no | input tokens mix, both directions |
| decoder **masked** self-attn | decoder | decoder | **yes** | *this is your GPT block* &mdash; no peeking ahead |
| decoder **cross-attn** (orange arrow) | **decoder** | **encoder** | no | each output token reads the input |

That orange arrow &mdash; **cross-attention** &mdash; is the one piece SmolVLA adds back on top of your GPT
(there, `K, V` come from the VLM features instead of an encoder). It is built in `02_smolvla_action_expert.ipynb`.


## 12. Scale it up — Karpathy's finale

Everything above trained a tiny model (~200K params, `block_size=32`) so it runs in seconds on a
CPU. The architecture is *identical* to the one in Karpathy's "Let's build GPT" finale — only the
knobs change. Here we dial them up to his final settings to get **recognizable Shakespeare**.

| knob          | tiny (above) | finale (below) |
|---------------|:------------:|:--------------:|
| `block_size`  | 32           | 256            |
| `batch_size`  | 16           | 64             |
| `n_embd`      | 64           | 384            |
| `n_heads`     | 4            | 6              |
| `n_layer`     | 4            | 6              |
| `dropout`     | 0.1          | 0.2            |
| `max_iters`   | 3000         | 5000           |

> **Run this on a GPU.** At ~10.7M params and `block_size=256` this is *very* slow on CPU
> (expect tens of minutes to hours). With the GPU speedups below (TF32 + on-device batching + bf16 autocast) it trains in
> roughly 15-25 min on a modest GPU and reaches
> **val loss ≈ 1.48**. `get_batch`/`estimate_loss` read the module-level `block_size` and
> `batch_size`, so we reassign those globals before rebuilding the model.


In [ ]:
# --- Karpathy's finale config (+ GPU speedups) --------------------------------
# Reassign the GLOBALS that get_batch() / estimate_loss() close over, then rebuild.
import time
block_size = 256      # max context length
batch_size = 64       # independent sequences per batch
n_embd, n_heads, n_layer, dropout = 384, 6, 6, 0.2
max_iters, eval_interval, lr = 5000, 500, 3e-4
log_interval = 100    # print a lightweight heartbeat this often (so it doesn't look frozen)

if device == "cpu":
    print("WARNING: no GPU detected — this config is very slow on CPU. "
          "Consider lowering max_iters or the model size, or running on a GPU.")
else:
    # Free ~2x speedup on Ampere/Ada GPUs: allow TF32 for matmuls / cudnn.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# SPEEDUP 1: keep the dataset ON the GPU and build batches with vectorized indexing.
# The original get_batch() sliced on the CPU and did a host->device copy every step,
# which starved the GPU (~29% util). Indexing on-device removes that per-step transfer.
train_data = train_data.to(device)
val_data = val_data.to(device)

def get_batch(split):
    """Fully-vectorized, on-device batch sampling (no Python loop, no host->device copy)."""
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,), device=device)   # (B,) start indices
    offs = torch.arange(block_size, device=device)                          # (T,)
    idx = ix[:, None] + offs[None, :]                                       # (B, T) gather indices
    x = d[idx]                                                              # (B, T) inputs
    y = d[idx + 1]                                                          # (B, T) targets (shift by 1)
    return x, y

# SPEEDUP 2: mixed precision. bfloat16 autocast on the GPU roughly halves step time
# and memory (bf16 keeps fp32's exponent range, so no GradScaler is needed).
amp_dtype = torch.bfloat16 if device == "cuda" else torch.float32

model = GPTLanguageModel(vocab_size, n_embd, n_heads, n_layer, block_size, dropout).to(device)
print("parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)  # AdamW: Adam with decoupled weight decay
t0 = time.time()
for it in range(max_iters + 1):
    # Periodically evaluate on train & val to track progress and detect overfitting
    if it % eval_interval == 0:
        losses = estimate_loss(model, eval_iters=200)
        print(f"step {it:4d} | train {losses['train']:.3f} | val {losses['val']:.3f}", flush=True)
    xb, yb = get_batch("train")              # sample a fresh random batch each step (on GPU)
    with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
        _, loss = model(xb, yb)              # forward pass -> compute cross-entropy loss (in bf16)
    optimizer.zero_grad(set_to_none=True)    # reset gradients (set_to_none is slightly faster)
    loss.backward()                          # backward pass -> compute gradients
    optimizer.step()                         # update parameters using AdamW
    # Heartbeat: proves the loop is alive between (expensive) evals and reports throughput.
    if it % log_interval == 0 and it > 0:
        dt = time.time() - t0
        print(f"  ..step {it:4d} | batch loss {loss.item():.3f} | {log_interval / dt:.1f} it/s", flush=True)
        t0 = time.time()


In [55]:
# Generate from the scaled-up model. With val loss ~1.48 this should read like
# (archaic, occasionally nonsensical) Shakespeare: real words, names, and dialogue structure.
start = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(start, max_new_tokens=2000)[0].tolist()))



I would bade amaze! work'st thou with sweet!
Away, Tybalt love, leave your want your father,
Advance me you pleased with lawful coat,
Duke down of life.

NORTHUMBERLAND:
My noble lord! I would buy good fath.
Lord he be known whither lies dead.

KING RICHARD III:
I not made the spier of my deither path;
But now the king and die sear crown. O, that
this shame, by revenge; but 'tis double;
strike your league.
Go, we are touch too forestiping friends!

QUEEN ELIZABETH:
We have heart of the prick-black, like what he babes!


QUEEN ELIZABETH:
My son, I pray, for this first is so.
Hath he the rich he did with his man part plack sea
From tanks. Did what swear she care
I'll not plump infeed, which linsul: be think'd so?
And from which owe's my dear attony, hark. Sweet low thee,
Alack, o'er side thou mock my bonishment,
To breath the my troublest way to the mean to put
The citizens, at wail and full's uspiec,
As law and do wrive Richclaim this aith.

GLOUCESTER:

NORTHUMBERLAND:
I speak no.
Wha